# UAV Flight Telemetry Analytics & Anomaly Screening Pipeline
---
This notebook demonstrates the end-to-end telemetry engineering and multi-detector anomaly screening workflow for unmanned aerial vehicles (UAVs) using `uavtel`.

### Pipeline Stages:
1. **Ingestion & Auto-Mapping**: Ingest PX4/ArduPilot CSV flight logs and normalize schema.
2. **Cleaning & Resampling**: Sort, deduplicate, convert units, and align all channels to a uniform 10 Hz grid.
3. **Feature Engineering**: Vectorized derivatives ($v_z$), rolling statistics, battery power ($P = V \times I$) & cumulative energy ($Wh$), and flight phase tagging.
4. **Baseline Anomaly Screening**: Multi-method detection (Rolling Z-Score, IQR/Tukey, EWMA, Rule-based, Isolation Forest, Weighted Ensemble).
5. **Benchmarking & Evaluation**: Point-wise & event-level precision, recall, F1, and detection latency against ground truth.
6. **Telemetry Visualizations**: Multi-channel plots, battery discharge curves, and 2D GPS trajectories.

In [1]:
import sys
from pathlib import Path
# Add core package to sys.path
sys.path.insert(0, str(Path.cwd().parent / 'core'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uavtel

print(f'uavtel version: {uavtel.__version__}')

## 1. Data Ingestion & Auto-Column Mapping
Load a simulated 15-minute flight log and verify canonical channel resolution.

In [2]:
raw_df, io_meta = uavtel.load_csv('../data/simulated_15min_flight.csv')
print(f'Loaded {len(raw_df)} telemetry rows across {len(raw_df.columns)} channels.')
print(f"Schema Valid: {io_meta['is_valid']}")
raw_df[['timestamp', 'alt', 'groundspeed', 'voltage', 'current', 'throttle']].head()

## 2. Telemetry Cleaning & 10 Hz Timebase Resampling
Align all sensor streams to a synchronized 10 Hz grid (dt = 0.1s) and report completeness.

In [3]:
clean_df, clean_meta = uavtel.clean_telemetry(raw_df, freq_hz=10.0)
print(f"Cleaned duration: {clean_meta['duration_sec']}s ({clean_meta['clean_rows']} samples)")
pd.DataFrame(clean_meta['missing_report']).head(6)

## 3. Feature Engineering & Flight Phase Classification
Compute vertical velocity $v_z$, electrical power ($W$), cumulative energy ($Wh$), and flight phases (`ground`, `climb`, `cruise`, `loiter`, `descent`).

In [4]:
feat_df = uavtel.extract_features(clean_df, freq_hz=10.0)
print(f'Extracted {len(feat_df.columns)} features.')
print(f"Total Energy Consumed: {feat_df['energy_wh'].iloc[-1]:.2f} Wh")
print('Flight Phase Distribution:')
print(feat_df['flight_phase'].value_counts(normalize=True).round(3) * 100)

## 4. Multi-Detector Anomaly Screening
Run the 6 screening detectors and merge continuous alarm flags into engineering review events.

In [5]:
detections, events = uavtel.run_all_detectors(feat_df, freq_hz=10.0)
print(f'Screened {len(events)} discrete anomaly events.')
df_events = pd.DataFrame(events)
df_events[['event_id', 'start_time', 'end_time', 'duration_sec', 'channel', 'severity', 'peak_score', 'description']].head(8)

## 5. Benchmarking Against Ground-Truth Anomalies
Compute precision, recall, F1-Score, and detection delay latency across all methods.

In [6]:
df_benchmark = uavtel.evaluate_all_methods(feat_df, detections)
df_benchmark

## 6. Multi-Channel Visualizations
Inspect the synchronized telemetry charts, battery curves, and spatial GPS tracks.

In [7]:
fig_multi = uavtel.plot_multichannel_timeseries(feat_df, events, figsize=(14, 9))
plt.show()

fig_bat = uavtel.plot_battery_trend(feat_df, figsize=(12, 4))
plt.show()

fig_gps = uavtel.plot_gps_track(feat_df, events, figsize=(7, 7))
plt.show()